# HW4: Momentum vs. LightGBM vs. the S&P 500 total-return index

This notebook runs the whole homework from top to bottom on the full `equities_top2000.duckdb`:

1. Load the data and build a point-in-time feature panel.
2. Screen 17 candidate features down to 12 with low correlation (2010–2012 only).
3. Validate LightGBM, Ridge and 12-1 momentum walk-forward on 2013–2015.
4. Run the final test (Apr 1, 2016 – Sep 30, 2026) with my own backtester ([hw4/backtest.py](hw4/backtest.py)).
5. Check the accounting and timing, and compare with `^SP500TR`.

**Submitted strategy:** 12-1 momentum, top 50 names every 10 sessions, 90-day lots. **LightGBM** (my original frozen model) and **Ridge** run under exactly the same trading rules as comparisons.

The code lives in the `hw4/` package and `run_hw4.py`; this notebook calls those functions so the numbers match the README. Runtime: about 8 minutes on a laptop, plus a few minutes the first time to build the cached panel in `outputs/`.

In [1]:
import json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / "run_hw4.py").exists():                    # allow running from the repo root
    HERE = HERE / "Homeworks" / "HW4" / "Shashank_Mishra"
sys.path.insert(0, str(HERE))
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.float_format", "{:,.4f}".format)

import run_hw4 as R
from hw4 import data, features as F, model as M
from hw4.backtest import BuyAndHold, performance

DB = data.find_database()          # Lectures/modeling/data/ first, then the repo root
print("database:", DB.name)

database: equities_top2000.duckdb


## 1. Data and the point-in-time panel

`load_everything` reads prices, membership, fundamentals and sectors from DuckDB (read-only) and builds one row per stock and feature date (every 5th session). The rules that prevent look-ahead:

* **Entry and exit.** Features use data through the signal close *t*. Entry is the next session's close, and the exit is the first quote on or after entry + 90 calendar days.
* **Universe.** A stock is eligible only if it is in the dated top-2000 list for the entry quarter.
* **Fundamentals.** These are used from `filing_date + 1` day.
* **Prices.** Bad prices are quarantined and nothing is forward-filled.

In [2]:
t0 = time.time()
md_, panel = R.load_everything(DB)
print(f"{len(md_.calendar):,} sessions x {len(md_.symbols):,} symbols; panel {len(panel):,} rows "
      f"({panel.date.min().date()} to {panel.date.max().date()}), {time.time()-t0:.0f}s")
panel[["date", "symbol", "universe_rank", "close", "mom_12_1", "earnings_yield", "entry_date", "label_end", "target"]].sample(5, random_state=0)

6,726 sessions x 3,849 symbols; panel 1,664,375 rows (2010-01-04 to 2026-09-23), 2s


,date,symbol,universe_rank,close,mom_12_1,earnings_yield,entry_date,label_end,target
504663,2015-02-11,CTRA,340.0000,28.0900,-0.2507,0.0344,2015-02-12,2015-05-13,0.2583
269562,2012-09-27,VOXX,"1,558.0000",7.5200,0.3658,0.1054,2012-09-28,2012-12-27,-0.1217
1414198,2024-03-19,KFY,"1,301.0000",63.4400,0.1935,0.0465,2024-03-20,2024-06-18,0.0386
1360863,2023-09-05,MEG,"1,871.0000",38.1800,-0.0453,-0.0335,2023-09-06,2023-12-05,-0.1676
262447,2012-09-06,FOSL,429.0000,85.4300,-0.0505,0.0571,2012-09-07,2012-12-06,0.0307


## 2. Features: 17 candidates, screened to 12 with low correlation

The screen uses only 2010–2012 rows whose 90-day label had already ended:

1. Rank the candidates by |rank IC|.
2. Keep a feature only if its average same-date |Spearman ρ| with every feature already kept is ≤ 0.4.

In [3]:
screen = panel.loc[panel.date.lt(R.SCREEN_END) & panel.label_end.lt(R.SCREEN_END) & panel.target_rank.notna()]
kept, corr, stats = F.select_low_correlation_features(screen, [f"r_{c}" for c in F.CANDIDATES], R.CORR_CUTOFF)
feats = [k[2:] for k in kept]
print("kept:", feats)
stats.index = stats.index.str[2:]
stats

kept: ['idio_vol_252', 'earnings_yield', 'debt_to_assets', 'sales_to_price', 'gross_profitability', 'earnings_change', 'mom_12_1', 'volume_trend', 'accruals', 'asset_growth', 'rev_1m', 'beta_252']


,mean_ic,ic_tstat,coverage,decision
idio_vol_252,-0.0735,-5.8532,0.9855,kept
vol_63,-0.0645,-4.5418,1.0000,dropped: |corr| 0.86 with r_idio_vol_252
earnings_yield,0.0549,8.0673,0.9077,kept
max_ret_21,-0.0504,-4.6118,1.0000,dropped: |corr| 0.66 with r_idio_vol_252
debt_to_assets,0.0457,9.9568,0.9962,kept
dist_52w_high,0.0387,3.0991,0.9858,dropped: |corr| 0.55 with r_idio_vol_252
log_mcap,0.0386,5.8733,0.9970,dropped: |corr| 0.65 with r_idio_vol_252
sales_to_price,0.0342,3.9334,0.9077,kept
gross_profitability,0.0337,5.5094,0.9077,kept
earnings_change,0.0316,7.3400,0.8790,kept


In [4]:
c = corr.loc[kept, kept]
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(c.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feats)), feats, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(feats)), feats, fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8); ax.set_title("Kept features: mean same-date rank correlation, 2010-2012")
plt.tight_layout(); plt.show()
print("largest remaining |rho|:", round(c.where(~np.eye(len(c), dtype=bool)).abs().max().max(), 3))

largest remaining |rho|: 0.384


## 3. Validation on 2013–2015

I use expanding-window folds: each model is fit just before the first signal of 2013, 2014 and 2015, on labels that had already ended, and then scores that year.

* **LightGBM:** 7 leaves, at least 5,000 rows per leaf, target = the winsorized, same-date-demeaned 90-day return.
* **Ridge:** fit on the same ranks.
* **Momentum:** no fitting; the score is the 12-1 momentum percentile.

**Rank IC** is the average same-date Spearman correlation between score and realized 90-day return.

In [5]:
dev = panel.loc[panel.date.le(R.DEV_END)]
sectors = sorted(panel.sector.unique())
NAMES = ["momentum_only", "lgb_excess", "ridge"]
folds = M.yearly_folds(dev, R.VALIDATION_YEARS)
val_preds, ic_rows = {n: [] for n in NAMES}, []
for y, fit_date, train, valid in folds:
    rows, cache = dev.loc[dev.date.dt.year.eq(y)], {}
    assert train.label_end.max() < fit_date
    for n in NAMES:
        s, _ = R.score(n, feats, sectors, train, rows, cache)
        val_preds[n].append(rows.assign(score=s))
        m = M.prediction_metrics(rows.assign(score=s).loc[rows.target.notna()], top_n=50)
        ic_rows.append({"year": y, "model": n, "rank_IC": m.mean_rank_IC, "top50_median_minus_all": m.iloc[4]})
    print(f"{y}: fit {fit_date.date()} on {len(train):,} rows, last label ended {train.label_end.max().date()}")
pd.DataFrame(ic_rows).pivot(index="model", columns="year", values="rank_IC").assign(mean=lambda x: x.mean(axis=1))

2013: fit 2013-01-03 on 269,617 rows, last label ended 2012-12-27


2014: fit 2014-01-08 on 370,835 rows, last label ended 2014-01-06


2015: fit 2015-01-06 on 470,301 rows, last label ended 2015-01-02


year,2013,2014,2015,mean
model,,,,
lgb_excess,0.0132,0.0092,0.0008,0.0077
momentum_only,0.0433,-0.0126,0.0697,0.0335
ridge,-0.0281,0.0549,0.0690,0.0319


Validation backtests, all with the same rules:
* every 10 sessions, buy the top 50 of the top-2000 universe;
* price ≥ $5, a bad-price screen, and the momentum-median screen;
* 6 overlapping cohorts, 10 bps per side.

In [6]:
cfg = json.loads((R.FINAL_DIR / "chosen_config.json").read_text()) if (R.FINAL_DIR / "chosen_config.json").exists() \
      else {"max_rank": 2000, "top_n": 50, "min_mom_rank": 0.5}
RULES = dict(max_rank=cfg["max_rank"], top_n=cfg["top_n"], min_mom_rank=cfg["min_mom_rank"])
sig = R.signal_dates(md_, folds[0][1], R.DEV_END)
start = md_.calendar[md_.calendar.get_loc(sig[0]) + 1]
val = {n: performance(R.run_strategy(md_, pd.concat(val_preds[n]), sig, start, R.DEV_END, **RULES)[0].daily.nav) for n in NAMES}
val["^SP500TR"] = performance(BuyAndHold.run(md_.benchmark, start, R.DEV_END).nav)
pd.DataFrame(val).loc[["final_value", "CAGR", "ann_vol", "sharpe_rf2", "max_drawdown"]]

,momentum_only,lgb_excess,ridge,^SP500TR
final_value,"2,016,715.6253","1,473,898.5471","1,389,805.3383","1,477,409.3533"
CAGR,0.2666,0.1396,0.1173,0.1405
ann_vol,0.2987,0.1514,0.1309,0.1277
sharpe_rf2,0.8592,0.8071,0.7605,0.9376
max_drawdown,-0.2084,-0.1386,-0.1374,-0.1204


## 4. Final test: Apr 1, 2016 – Sep 30, 2026

LightGBM and Ridge are refit once a year, on the first signal date on or after March 31, with an expanding window from 2010. Every fit asserts that its last training label ended before the fit date. Momentum needs no fit.

In [7]:
sig = R.signal_dates(md_, R.TEST_START - pd.Timedelta(days=1), R.TEST_END)
assert sig[0] == pd.Timestamp("2016-03-31")
fit_dates = sorted({sig[sig >= pd.Timestamp(f"{y}-03-31")][0] for y in range(2016, 2027)})
preds, refits = {n: [] for n in NAMES}, []
for k, fd in enumerate(fit_dates):
    nxt = fit_dates[k + 1] if k + 1 < len(fit_dates) else R.TEST_END + pd.Timedelta(days=1)
    train = M.training_rows(panel, fd)
    assert train.label_end.max() < fd, "training label not yet observed"
    rows, cache = panel.loc[panel.date.isin(sig[(sig >= fd) & (sig < nxt)])], {}
    for n in NAMES:
        preds[n].append(rows.assign(score=R.score(n, feats, sectors, train, rows, cache)[0]))
    refits.append({"fit_date": fd.date(), "train_rows": len(train), "last_label_end": train.label_end.max().date()})
preds = {n: pd.concat(v) for n, v in preds.items()}
pd.DataFrame(refits)

,fit_date,train_rows,last_label_end
0,2016-03-31,593835,2016-03-28
1,2017-04-11,697322,2017-04-10
2,2018-04-10,796702,2018-04-05
3,2019-04-08,896070,2019-04-04
4,2020-04-03,995320,2020-04-01
5,2021-04-01,1094293,2021-03-29
6,2022-04-12,1196119,2022-04-11
7,2023-04-12,1293758,2023-04-10
8,2024-04-10,1391712,2024-04-09
9,2025-04-09,1489312,2025-04-07


In [8]:
runs = {n: R.run_strategy(md_, preds[n], sig, R.TEST_START, R.TEST_END, **RULES) for n in NAMES}
bench = BuyAndHold.run(md_.benchmark, R.TEST_START, R.TEST_END)
perf = pd.DataFrame({"Momentum (submitted)": performance(runs["momentum_only"][0].daily.nav),
                     "^SP500TR": performance(bench.nav),
                     "LightGBM (comparison)": performance(runs["lgb_excess"][0].daily.nav),
                     "Ridge (comparison)": performance(runs["ridge"][0].daily.nav)})
perf.loc["excess_CAGR_vs_SP500TR"] = perf.loc["CAGR"] - perf.at["CAGR", "^SP500TR"]
perf.loc[["final_value", "CAGR", "ann_vol", "sharpe_rf2", "max_drawdown", "excess_CAGR_vs_SP500TR"]]

,Momentum (submitted),^SP500TR,LightGBM (comparison),Ridge (comparison)
final_value,"4,824,110.4486","4,402,746.7763","2,521,504.3889","2,814,760.7535"
CAGR,0.1617,0.1517,0.0921,0.1036
ann_vol,0.2765,0.1782,0.2218,0.1704
sharpe_rf2,0.6107,0.7725,0.4200,0.5490
max_drawdown,-0.3941,-0.3379,-0.4111,-0.3075
excess_CAGR_vs_SP500TR,0.0101,0.0000,-0.0596,-0.0480


In [9]:
res = runs["momentum_only"][0]
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(res.daily.nav / 1e6, label="Momentum, top 50 (submitted)", lw=1.8, color="#1f5aa6")
ax.plot(bench.nav / 1e6, label="^SP500TR buy & hold", lw=1.8, color="#c0392b")
ax.plot(runs["lgb_excess"][0].daily.nav / 1e6, label="LightGBM (same rules)", lw=1, ls="--", color="#7f8c8d")
ax.plot(runs["ridge"][0].daily.nav / 1e6, label="Ridge (same rules)", lw=1, ls="--", color="#b9a14a")
ax.set_ylabel(r"Portfolio value (\$ millions, from \$1M)"); ax.set_title("Final test: Apr 1, 2016 - Sep 30, 2026")
ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()
beat = perf.at["final_value", "Momentum (submitted)"] - perf.at["final_value", "^SP500TR"]
print(f"Momentum {'beat' if beat > 0 else 'trailed'} the benchmark by ${abs(beat):,.0f} "
      f"({perf.at['excess_CAGR_vs_SP500TR', 'Momentum (submitted)']*100:+.2f} pp CAGR)")

Momentum beat the benchmark by $421,364 (+1.01 pp CAGR)


## 5. Accounting and timing checks

`accounting_checks` re-derives these from the trade log, independently of the backtester's own bookkeeping:
* cash and NAV;
* next-session fills;
* first-quote exits;
* fill prices, re-queried directly from DuckDB;
* the value of lots whose stock stopped quoting.

The hand-calculated unit tests are in `tests/test_backtest.py` (run `python -m pytest tests -q`).

In [10]:
checks = R.accounting_checks(res, md_, DB, sig)

{
  "cash_reconciles": true,
  "min_cash": 0.0,
  "nav_reconciles": true,
  "nav_equals_cash_plus_holdings": true,
  "buys_fill_next_session": true,
  "all_signal_dates_on_grid": true,
  "no_sale_before_maturity": true,
  "sales_at_first_quote_on_or_after_maturity": true,
  "sessions_late_distribution": {
    "0.0": 12678,
    "1.0": 1,
    "3.0": 1
  },
  "sampled_fill_prices_match_db": true,
  "fills": {
    "buys": 13096,
    "sells": 12680,
    "open_lots_at_end": 416
  },
  "fees_paid": 159630.77147774026,
  "skipped_orders_no_entry_quote": 4,
  "final_nav_if_unquoted_lots_written_to_zero": 3905990.1495219776,
  "worked_example": {
    "symbol": "MXL",
    "signal": "2016-03-31",
    "entry": "2016-04-01",
    "due": "2016-06-30",
    "exit": "2016-06-30",
    "entry_price": 18.62,
    "units": 178.8401358755816,
    "entry_cost_incl_fee": 3333.333333333333,
    "exit_price": 17.98,
    "exit_proceeds_after_fee": 3212.3300973999144,
    "realized_pnl": -121.00323593341861,
    "ch

In [11]:
# One lot traced by hand: units = budget / (price * 1.001); proceeds = units * exit * 0.999.
ex = checks["worked_example"]
units = ex["entry_cost_incl_fee"] / (ex["entry_price"] * 1.001)
proceeds = units * ex["exit_price"] * 0.999
print(f"{ex['symbol']}: signal {ex['signal']} -> buy {ex['entry']} @ {ex['entry_price']} -> due {ex['due']} -> sell {ex['exit']} @ {ex['exit_price']}")
print(f"units {units:.4f} (backtester {ex['units']:.4f}); PnL by hand {proceeds - ex['entry_cost_incl_fee']:.2f} (backtester {ex['realized_pnl']:.2f})")
stuck = pd.DataFrame(checks.get("open_lots_without_final_quote", []))
print(f"open lots without a final quote: {len(stuck)} worth ${stuck.market_value.sum():,.0f}; "
      f"NAV if written to zero: ${checks['final_nav_if_unquoted_lots_written_to_zero']:,.0f}")

MXL: signal 2016-03-31 -> buy 2016-04-01 @ 18.62 -> due 2016-06-30 -> sell 2016-06-30 @ 17.98
units 178.8401 (backtester 178.8401); PnL by hand -121.00 (backtester -121.00)
open lots without a final quote: 119 worth $918,120; NAV if written to zero: $3,905,990


## 6. What the momentum book actually holds

Momentum is not a proxy for the index. It buys each period's strongest 12-month winners, equally weighted, from the whole top-2000 universe, and that list is mostly mid and small caps.

In [12]:
f = res.fills
buys = f.loc[f.side.eq("BUY")].merge(panel[["date", "symbol", "universe_rank", "sector"]],
                                     left_on=["signal_date", "symbol"], right_on=["date", "symbol"], how="left")
print(f"median buy size rank: {buys.universe_rank.median():.0f}; share of buys from the top 500: {(buys.universe_rank <= 500).mean():.0%}")
print("sector mix of buys:", buys.sector.value_counts(normalize=True).head(5).round(2).to_dict())
pnl = f.loc[f.side.eq("SELL")].groupby("symbol").realized_pnl.sum().sort_values()
pd.concat([pnl.head(5), pnl.tail(5)]).rename("realized PnL ($)").to_frame()

median buy size rank: 1230; share of buys from the top 500: 16%
sector mix of buys: {'Healthcare': 0.24, 'Technology': 0.21, 'Industrials': 0.12, 'Consumer Cyclical': 0.11, 'Energy': 0.09}


,realized PnL ($)
symbol,
RUBY,"-37,498.0279"
AMRS,"-37,382.4462"
PACB,"-36,888.5706"
WULF,"-35,699.2742"
RIOT,"-35,060.4634"
VKTX,"117,988.4202"
SMCI,"120,036.9802"
BE,"123,365.2686"
APP,"141,832.9937"


In [13]:
yr = lambda nav: (nav.resample("YE").last().pct_change().fillna(nav.resample("YE").last().iloc[0] / 1e6 - 1) * 100).round(1)
pd.DataFrame({"Momentum": yr(res.daily.nav), "^SP500TR": yr(bench.nav * 1.0),
              "LightGBM": yr(runs["lgb_excess"][0].daily.nav)}).set_axis(range(2016, 2027)).T

,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Momentum,13.9000,22.6000,-15.3000,26.8000,55.9000,11.2000,-9.5000,19.2000,23.7000,20.6000,15.2000
^SP500TR,9.8000,21.8000,-4.4000,31.5000,18.4000,28.7000,-18.1000,26.3000,25.0000,17.9000,12.7000
LightGBM,12.3000,23.5000,-16.9000,12.6000,62.2000,8.7000,-20.0000,16.7000,7.8000,8.8000,0.5000


## 7. Findings

* **Result.** Momentum turned \$1M into about \$4.82M (16.2% CAGR), against \$4.40M (15.2%) for `^SP500TR`. That beats the index by about 1 pp a year, but with higher volatility (28% vs. 18%), a lower Sharpe (0.61 vs. 0.77) and a deeper drawdown (−39% vs. −34%, Feb–Mar 2020).
* **How momentum was chosen.**
  * My original plan, frozen before the test, was LightGBM: 9.2% CAGR.
  * Momentum ran beside it as a pre-declared baseline and had the best validation CAGR (26.7%).
  * Choosing momentum as the final strategy was decided *after* the test, so the 1 pp edge is not a clean out-of-sample result. Its setup (top 2000, 50 names) is the one with the best momentum validation Sharpe.
* **LightGBM.** It was positive but weak. Its rank IC was small, and its top picks leaned toward value stocks, which had a poor decade. Ridge did about as well as LightGBM.
* **Data gaps.** About \$0.9M of final value sits in lots whose stock stopped quoting (acquisitions, renames). The database has no delisting proceeds, so they are carried at their last price; written to zero, NAV would be about \$3.9M.
* **Next steps.**
  * Longer validation history.
  * Decide in advance between cap-weighting and a large-cap universe.
  * Recover deal proceeds for delisted lots.
  * A volume-based cost model.